# Polygon Trade Analysis

Loads raw per-side trades from `data/trades_polygon/*.parquet` and markets from
`data/markets/*.parquet`, then performs wallet-level P&L analysis and extracts trades
of the **top 5% wallets** identified on the training period.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.


In [25]:
import json
import datetime
from pathlib import Path

import pandas as pd

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

## Parameters

In [26]:
# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
# Trades on tokens from markets outside this range are excluded.
START_DATE = datetime.date(2026, 3, 1)
END_DATE   = datetime.date(2026, 3, 11)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-5% wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 3, 7)

# ── input paths ───────────────────────────────────────────────────────────────
TRADES_DIR  = Path("../../data/trades_polygon")
MARKETS_DIR = Path("../../data/markets")

# ── output directory (partitioned parquet shards) ────────────────────────────
OUT_DIR = Path("../../data/polygon_trades_processed")


## 1 · Load markets

Each `data/markets/YYYY-MM.parquet` file has three columns:
- `end_date_iso` – market resolution date string
- `condition_id` – unique market identifier
- `market_json` – full market definition as a JSON string

We parse every `market_json` to build:
1. **token_lookup** – maps `token_id → {condition_id, outcome, token_winner, final_price}`
2. **market_meta** – per-market DataFrame with `condition_id`, `question`, `end_date`, `market_slug`

In [27]:
# Load only the market files whose month falls within [START_DATE, END_DATE].
# [:1] keeps execution fast; increase the slice to load more markets.
import datetime as _dt
def _file_in_range(p, start, end):
    """Return True if YYYY-MM.parquet filename falls within the date range."""
    try:
        y, m = (int(x) for x in p.stem.split("-"))
        file_date = _dt.date(y, m, 1)
        return start.replace(day=1) <= file_date <= end.replace(day=1)
    except Exception:
        return False

market_files = [
    p for p in sorted(MARKETS_DIR.glob("*.parquet"))
    if _file_in_range(p, START_DATE, END_DATE)
]
print(f"Found {len(market_files)} market file(s)")

all_market_rows = pd.concat(
    [pd.read_parquet(f) for f in market_files],
    ignore_index=True,
)
# deduplicate by condition_id (keep first occurrence)
all_market_rows = all_market_rows.drop_duplicates(subset="condition_id", keep="first")
print(f"Unique markets (all):  {len(all_market_rows):,}")

# ── apply START_DATE / END_DATE filter ───────────────────────────────────────
# Parse end_date_iso as a date; keep only markets whose resolution date falls
# within [START_DATE, END_DATE] (inclusive).  Markets outside this range are
# excluded, and their tokens will not appear in the token_lookup, so any trades
# on those tokens will be dropped during the enrichment join.
end_dates = pd.to_datetime(all_market_rows["end_date_iso"], utc=True, errors="coerce").dt.date
in_range  = (end_dates >= START_DATE) & (end_dates <= END_DATE)
all_market_rows = all_market_rows[in_range].reset_index(drop=True)
print(f"Unique markets (filtered {START_DATE} → {END_DATE}): {len(all_market_rows):,}")
all_market_rows.head(2)

Found 1 market file(s)
Unique markets (all):  95,876
Unique markets (filtered 2026-03-01 → 2026-03-11): 58,307


,end_date_iso,condition_id,market_json
0,2026-03-01T00:00:00Z,0x00127f7be65484384710f01b53e56d59bacf6433c5c85c80c3c7e692d4d57239,"{""enable_order_book"":false,""active"":true,""closed"":true,""archived"":false,""accepting_orders"":false,""accepting_order_timestamp"":""2026-01-29T02:36:54Z"",""minimum_order_size"":5,""minimum_tick_size"":0.001,""condition_id"":""0x00127f7be65484384710f01b53e56d59bacf6433c5c85c80c3c7e692d4d57239"",""question_id"":""0x4776b0425df4363baa8a35a33780c038839d2c5f199ebe1e396e03ee70652eb0"",""question"":""Will Nikola Jokic Rookie Card be above $215 by end of February?"",""description"":""This market will resolve to \""Yes\"" if the ungraded price of the Nikola Jokic #335 Rookie (2015 Panini Prizm) sports card on pricecharting.com is strictly greater than the price specified in the title on the last day of February 2026. Otherwise, this market will resolve to \""No\"".\n\nThe resolution source for this market is pricecharting.com, specifically the \""Ungraded\"" price available at https://www.sportscardspro.com/game/basketball-cards-2015-panini-prizm/nikola-jokic-335.\n\nThe price for a given month becomes final at the end of that calendar month (midnight UTC) and is displayed on the chart the following day.\n\nPlease note that this market is about the ungraded price according to pricecharting.com, not graded prices or prices from other sources."",""market_slug"":""nikola-jokic-rookie-card-above-215-on-february-28-2026"",""end_date_iso"":""2026-03-01T00:00:00Z"",""game_start_time"":null,""seconds_delay"":0,""fpmm"":"""",""maker_base_fee"":0,""taker_base_fee"":0,""notifications_enabled"":true,""neg_risk"":false,""neg_risk_market_id"":"""",""neg_risk_request_id"":"""",""icon"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/jokic-rookie-7c6442b4ce.jpg"",""image"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/jokic-rookie-7c6442b4ce.jpg"",""rewards"":{""rates"":null,""min_size"":50,""max_spread"":4.5},""is_50_50_outcome"":false,""tokens"":[{""token_id"":""18311785170204388506088988398848684615891091738926330363740178957819259908301"",""outcome"":""Yes"",""price"":1,""winner"":true},{""token_id"":""43435255201305657647993695058693314017600203047595086240653789695408239972490"",""outcome"":""No"",""price"":0,""winner"":false}],""tags"":[""Sports"",""Recurring"",""Monthly"",""Hide From New"",""Multi Strikes"",""Collectibles"",""Trading Cards""]}"
1,2026-03-01T00:00:00Z,0x0021181af68af2fa36b5b3a6189b0828805c64ac8adee0ac5e551f055db1e590,"{""enable_order_book"":false,""active"":true,""closed"":true,""archived"":false,""accepting_orders"":false,""accepting_order_timestamp"":""2026-02-28T16:22:03Z"",""minimum_order_size"":5,""minimum_tick_size"":0.01,""condition_id"":""0x0021181af68af2fa36b5b3a6189b0828805c64ac8adee0ac5e551f055db1e590"",""question_id"":""0xd79ad90167bf97f9248fb029ef778a8c48bcd4712125aff1baf2a0f8d05ac684"",""question"":""XRP Up or Down - March 1, 11:15AM-11:20AM ET"",""description"":""This market will resolve to \""Up\"" if the XRP price at the end of the time range specified in the title is greater than or equal to the price at the beginning of that range. Otherwise, it will resolve to \""Down\"".\nThe resolution source for this market is information from Chainlink, specifically the XRP/USD data stream available at https://data.chain.link/streams/xrp-usd.\nPlease note that this market is about the price according to Chainlink data stream XRP/USD, not according to other sources or spot markets."",""market_slug"":""xrp-updown-5m-1772381700"",""end_date_iso"":""2026-03-01T00:00:00Z"",""game_start_time"":null,""seconds_delay"":0,""fpmm"":"""",""maker_base_fee"":1000,""taker_base_fee"":1000,""notifications_enabled"":true,""neg_risk"":false,""neg_risk_market_id"":"""",""neg_risk_request_id"":"""",""icon"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/XRP-logo.png"",""image"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/XRP-logo.png"",""rewards"":{""rates"":null,""min_si

In [28]:
def build_lookups(market_rows: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """Parse market_json column and return (token_lookup, market_meta_df)."""
    token_lookup: dict[str, dict] = {}
    meta_rows: list[dict] = []

    for _, row in market_rows.iterrows():
        try:
            m = json_loads(row["market_json"])
        except Exception:
            continue

        cid = str(m.get("condition_id", row["condition_id"]))

        # ── token resolution lookup ───────────────────────────────────────────
        for tok in m.get("tokens", []):
            token_id = str(tok["token_id"])
            winner = bool(tok.get("winner", False))
            # final_price is determined solely by the token winner flag:
            # winner=True  → settled at 1.0 USDC per share
            # winner=False → settled at 0.0 USDC per share
            final_price = 1.0 if winner else 0.0
            token_lookup[token_id] = {
                "condition_id": cid,
                "outcome":      tok.get("outcome", ""),
                "token_winner": winner,
                "final_price":  final_price,
            }

        # ── market metadata ───────────────────────────────────────────────────
        meta_rows.append({
            "condition_id": cid,
            "question":     m.get("question", ""),
            "end_date":     pd.to_datetime(m.get("end_date_iso"), utc=True, errors="coerce"),
            "market_slug":  m.get("market_slug", ""),
        })

    market_meta = pd.DataFrame(meta_rows)
    return token_lookup, market_meta


token_lookup, market_meta = build_lookups(all_market_rows)
print(f"Token lookup entries: {len(token_lookup):,}")
print(f"Market meta rows:     {len(market_meta):,}")

Token lookup entries: 116,421
Market meta rows:     58,307


## 2 · Process partitioned per-side trades and filter to in-range markets

Trades are already stored as wallet-perspective rows (`wallet`, `side`) and
`TRADES_DIR` is partitioned, so we process each parquet shard independently to
avoid loading all fills into memory at once. For each shard we:
1. Inner-join with token resolution (`token_winner`, `final_price`)
2. Build `final_value_usdc = quantity × final_price`
3. Aggregate to wallet × market × timestamp and accumulate global results


In [29]:
trade_files = sorted(TRADES_DIR.glob("*.parquet"))[:1]
if not trade_files:
    raise FileNotFoundError(f"No parquet files found in {TRADES_DIR}")

print(f"Found {len(trade_files)} trade parquet file(s)")

sample_raw = pd.read_parquet(trade_files)
print(f"Sample shard rows ({trade_files[0].name}): {len(sample_raw):,}")
print(f"Columns: {sample_raw.columns.tolist()}")
if "trade_date" in sample_raw.columns:
    print(f"Sample shard date range: {sample_raw['trade_date'].min()} → {sample_raw['trade_date'].max()}")
sample_raw.head(3)


Found 1 trade parquet file(s)
Sample shard rows (0.parquet): 5,498,880
Columns: ['tx_hash', 'log_index', 'block_number', 'block_timestamp', 'trade_date', 'condition_id', 'token_id', 'outcome', 'wallet', 'side', 'price', 'quantity', 'usdc_amount', 'question', 'slug', 'fill_count', 'position', 'flipped', 'split', 'tags', 'raw_side', 'raw_outcome', 'raw_token_id', 'raw_price', 'raw_usdc_amount']
Sample shard date range: 2025-01-01 → 2026-03-11


,tx_hash,log_index,block_number,block_timestamp,trade_date,condition_id,token_id,outcome,wallet,side,...,fill_count,position,flipped,split,tags,raw_side,raw_outcome,raw_token_id,raw_price,raw_usdc_amount
0,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e3ddf9c6c7100eacb34fc,936,66158948,1735689665,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcdf60247b994f132ab79bb,78753205165658130534456351077321496563862891920229742523513427553266682271361,No,0x126f15c7bd21bf5bf136f3629e10990dc0677137,BUY,...,1,10.0,False,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 2025 Predictions]",BUY,No,78753205165658130534456351077321496563862891920229742523513427553266682271361,0.62,6.2
1,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e3ddf9c6c7100eacb34fc,936,66158948,1735689665,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcdf60247b994f132ab79bb,58665055277986534416719939405914621458010137331379938342097742389618466217100,Yes,0x3978a99e028eb590c98d8e9ddbe513298fa92f24,BUY,...,1,10.0,True,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 2025 Predictions]",SELL,No,78753205165658130534456351077321496563862891920229742523513427553266682271361,0.62,6.2
2,0xc12324352ff1897431a9cdd5e91bcc05cc7c28e88c38fce30001acc0181bf112,1531,66159089,1735689997,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcdf60247b994f132ab79bb,58665055277986534416719939405914621458010137331379938342097742389618466217100,Yes,0x260d1ae03c985f7c78ad286b5da14940b4739679,BUY,...,1,30.0,False,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 2025 Predictions]",BUY,Yes,58665055277986534416719939405914621458010137331379938342097742389618466217100,0.37,11.1


In [30]:
# ── build token resolution DataFrame from lookup ─────────────────────────────
token_df = pd.DataFrame.from_dict(token_lookup, orient="index")
token_df.index.name = "token_id"
token_df.reset_index(inplace=True)
token_df["token_id"] = token_df["token_id"].astype(str)

def enrich_trade_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    """Attach settlement columns and computed datetime/value to one shard."""
    if chunk.empty:
        return chunk

    c = chunk.copy()
    c["token_id"] = c["token_id"].astype(str)
    c = c.merge(
        token_df[["token_id", "token_winner", "final_price"]],
        on="token_id",
        how="inner",
    )
    if c.empty:
        return c

    c["dt"] = pd.to_datetime(c["block_timestamp"], unit="s", utc=True)
    c["final_value_usdc"] = c["quantity"] * c["final_price"]
    return c

GROUP_KEYS = ["wallet", "condition_id", "dt"]

READ_COLS = [
    "tx_hash", "log_index", "block_timestamp", "trade_date", "condition_id",
    "token_id", "outcome", "price", "quantity", "usdc_amount", "position",
    "wallet", "side",
]

grouped_parts: list[pd.DataFrame] = []
sample_fills = None

total_raw_rows = 0
total_filtered_rows = 0

for i, f in enumerate(trade_files, start=1):
    raw_chunk = pd.read_parquet(f, columns=READ_COLS)
    total_raw_rows += len(raw_chunk)

    enriched_chunk = enrich_trade_chunk(raw_chunk)
    total_filtered_rows += len(enriched_chunk)

    if sample_fills is None and len(enriched_chunk) > 0:
        sample_fills = enriched_chunk.head(5).copy()

    if len(enriched_chunk) == 0:
        continue

    grouped_part = (
        enriched_chunk.groupby(GROUP_KEYS, sort=False)
        .agg(
            side             = ("side",             "first"),
            outcome          = ("outcome",          "first"),
            token_winner     = ("token_winner",     "first"),
            final_price      = ("final_price",      "first"),
            position         = ("position",         "max"),
            total_quantity   = ("quantity",         "sum"),
            price_sum        = ("price",            "sum"),
            price_count      = ("price",            "count"),
            trade_value_usdc = ("usdc_amount",      "sum"),
            final_value_usdc = ("final_value_usdc", "sum"),
            num_fills        = ("tx_hash",          "count"),
        )
        .reset_index()
    )
    grouped_parts.append(grouped_part)

    if i % 25 == 0 or i == len(trade_files):
        print(
            f"Processed {i}/{len(trade_files)} shards | "
            f"raw rows: {total_raw_rows:,} | in-range rows: {total_filtered_rows:,}"
        )

if not grouped_parts:
    raise ValueError("No in-range trade rows after token filter")

grouped = pd.concat(grouped_parts, ignore_index=True)

# In case a key spans multiple shards, combine partial aggregates.
grouped = (
    grouped.groupby(GROUP_KEYS, sort=False)
    .agg(
        side             = ("side",             "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("total_quantity",   "sum"),
        price_sum        = ("price_sum",        "sum"),
        price_count      = ("price_count",      "sum"),
        trade_value_usdc = ("trade_value_usdc", "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("num_fills",        "sum"),
    )
    .reset_index()
)

grouped["avg_price"] = grouped["price_sum"] / grouped["price_count"]
grouped.drop(columns=["price_sum", "price_count"], inplace=True)
grouped.sort_values(["wallet", "condition_id", "dt"], inplace=True, ignore_index=True)

print(
    f"Rows after market filter: {total_filtered_rows:,}  "
    f"(dropped {total_raw_rows - total_filtered_rows:,} rows outside [{START_DATE}, {END_DATE}])"
)
print(f"Prepared grouped rows: {len(grouped):,}")
if sample_fills is not None:
    sample_fills.head(5)


Processed 1/1 shards | raw rows: 5,498,880 | in-range rows: 657,989
Rows after market filter: 657,989  (dropped 4,840,891 rows outside [2026-03-01, 2026-03-11])
Prepared grouped rows: 455,437


## 3 · Enriched fill table

All fills are already filtered to in-range markets and carry resolution columns
from the inner join performed in the previous step.

| column | description |
|---|---|
| `token_winner` | `True` if the traded token is the winning outcome |
| `final_price` | Settlement price: 1.0 if `token_winner=True`, 0.0 otherwise |
| `final_value_usdc` | `quantity × final_price` — USDC value of the position at settlement |

In [31]:
if sample_fills is None:
    print("No enriched fills available after filtering")
    pd.DataFrame(columns=[
        "wallet", "side", "condition_id", "dt", "outcome", "quantity", "price",
        "position", "usdc_amount", "token_winner", "final_price", "final_value_usdc"
    ])
else:
    print(f"Raw rows scanned across all shards: {total_raw_rows:,}")
    print(f"In-range rows after token join:     {total_filtered_rows:,}")
    print(f"token_winner null: {sample_fills['token_winner'].isna().sum():,}  (sample should be 0)")
    sample_fills[["wallet", "side", "condition_id", "dt", "outcome", "quantity", "price", "position",
                  "usdc_amount", "token_winner", "final_price", "final_value_usdc"]].head(5)


Raw rows scanned across all shards: 5,498,880
In-range rows after token join:     657,989
token_winner null: 0  (sample should be 0)


## 4 · Group by wallet × market × timestamp

A single on-chain transaction can generate multiple fills at the same timestamp.
Each row in `grouped` aggregates all fills for one wallet in one market at one timestamp.

In [32]:
print(f"Grouped rows: {len(grouped):,}")
grouped.head(10)


Grouped rows: 455,437


,wallet,condition_id,dt,side,outcome,token_winner,final_price,position,total_quantity,trade_value_usdc,final_value_usdc,num_fills,avg_price
0,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x03e38f4c5abeb1560d3af133186dd3130f80183364964abf4724efbc7debeefe,2026-03-03 01:25:43+00:00,BUY,Clippers,True,1.0,10.000000,10.000000,5.1000,10.000,1,0.510000
1,0x00000db3f2fb2086866753b6e41e426a5312e225,0x06dba2eea29f5a283939f87dfb04d50eaeb35f8bad761ad16037ded5a8a2cfc5,2026-03-01 22:57:31+00:00,BUY,Michigan State Spartans,True,1.0,50.000000,50.000000,49.9500,50.000,1,0.999000
2,0x00000db3f2fb2086866753b6e41e426a5312e225,0x0b8c63df9d7e33b5dd98116f4c04e2eaf7e83df5d1e2e3368f5f5723f5b2e7ac,2026-03-01 23:14:27+00:00,BUY,Nets,True,1.0,50.000000,50.000000,49.4000,50.000,3,0.983333
3,0x0000769690190af200538d3b39f3af22ad2d296f,0x0f02a01099649c28962633ea6d0ad1db994be598089be058ee3366c3df3ed18c,2026-03-03 12:49:17+00:00,BUY,FORZE Reload,False,0.0,132.045625,132.045625,20.0000,0.000,5,0.152000
4,0x0001c7d116fd33b0edc523b56d36d9a2f47b93e0,0x07ac20a236c76c5d849be442162c57fc5201df6dbda8046d5d917d28c622888a,2026-03-08 00:04:53+00:00,BUY,Pistons,False,0.0,18.300000,18.300000,10.7970,0.000,2,0.590000
5,0x000269109ef14628e4a4a406acc548487ff1103e,0x01bded8d4eb4719ffcf3b3fd491f30f9726d643cce51f60e9a8b4423c8558da6,2026-03-03 19:57:23+00:00,BUY,paiN,True,1.0,425.289000,425.289000,382.7601,425.289,1,0.900000
6,0x000269109ef14628e4a4a406acc548487ff1103e,0x01bded8d4eb4719ffcf3b3fd491f30f9726d643cce51f60e9a8b4423c8558da6,2026-03-03 19:57:29+00:00,BUY,paiN,True,1.0,475.289000,50.000000,45.0000,50.000,1,0.900000
7,0x000269109ef14628e4a4a406acc548487ff1103e,0x01bded8d4eb4719ffcf3b3fd491f30f9726d643cce51f60e9a8b4423c8558da6,2026-03-03 19:57:41+00:00,BUY,paiN,True,1.0,487.189000,11.900000,10.7100,11.900,1,0.900000
8,0x000269109ef14628e4a4a406acc548487ff1103e,0x01bded8d4eb4719ffcf3b3fd491f30f9726d643cce51f60e9a8b4423c8558da6,2026-03-03 19:59:37+00:00,BUY,paiN,True,1.0,624.999000,137.810000,124.0290,137.810,1,0.900000
9,0x000269109ef14628e4a4a406acc548487ff1103e,0x01bded8d4eb4719ffcf3b3fd491f30f9726d643cce51f60e9a8b4423c8558da6,2026-03-03 20:02:19+00:00,SELL,paiN,True,1.0,0.009000,624.990000,618.7401,624.990,1,0.990000


## 5 · Per-trade P&L and wallet summary (training data only)

P&L is calculated **per trade** based on whether the traded token is a winner:

| side | `trade_pnl` formula | intuition |
|---|---|---|
| BUY | `final_value_usdc − trade_value_usdc` | bought tokens; profit if `final_price > entry_price` |
| SELL | `trade_value_usdc − final_value_usdc` | sold tokens; profit if sold above final settlement price |

where `final_value_usdc = total_quantity × final_price` (1.0 if the token is a winner, 0.0 otherwise).

Wallet P&L = `Σ trade_pnl` over training trades.

In [33]:
# ── per-trade P&L: final_value vs. trade_value, signed by side ──────────────
# BUY:  wallet paid trade_value_usdc, ends up with tokens worth final_value_usdc
#       → pnl = final_value_usdc - trade_value_usdc  (positive if token wins)
# SELL: wallet received trade_value_usdc, gave up tokens worth final_value_usdc
#       → pnl = trade_value_usdc - final_value_usdc  (positive if token loses)
is_buy = grouped["side"] == "BUY"
grouped["trade_pnl"] = (
    (grouped["final_value_usdc"] - grouped["trade_value_usdc"]).where(is_buy,
    grouped["trade_value_usdc"] - grouped["final_value_usdc"])
)

# ── restrict to training period for wallet ranking ────────────────────────────
end_train_ts   = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train  = grouped[grouped["dt"] < end_train_ts]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets      = ("condition_id",   "nunique"),
        num_trades       = ("num_fills",       "sum"),
        total_cost_usdc  = ("trade_value_usdc", "sum"),
        pnl_usdc         = ("trade_pnl",       "sum")
                )
    .sort_values("pnl_usdc", ascending=False)
    .reset_index()
)

print(f"Unique wallets (training period): {len(wallet_summary):,}")
wallet_summary.head(5)

Unique wallets (training period): 29,263


,wallet,num_markets,num_trades,total_cost_usdc,pnl_usdc
0,0x492442eab586f242b53bda933fd5de859c8a3782,7,71,737878.505300,357999.424700
1,0x07b8e44b90cc3e91b8d5fe60ea810d2534638e25,4,778,505054.688375,341776.408961
2,0x6a72f61820b26b1fe4d956e17b6dc2a1ea3033ee,11,844,372679.258635,341686.723958
3,0xc6b6ffce6c52b6fdf4bad57de303939f19ab13ac,1,8,216340.330000,216340.330000
4,0xc257ea7e3a81ca8e16df8935d44d513959fa358e,15,976,749004.824395,210405.677376


## 6 · Market-level volume summary

In [34]:
# join market metadata (question, end_date) for display
grouped_meta = grouped.merge(
    market_meta[["condition_id", "question", "end_date", "market_slug"]],
    on="condition_id",
    how="left",
)

market_summary = (
    grouped_meta.groupby(["condition_id", "question", "end_date"])
    .agg(
        num_wallets      = ("wallet",           "nunique"),
        num_trades       = ("num_fills",         "sum"),
        volume_usdc      = ("trade_value_usdc",  "sum"),
        final_value_usdc = ("final_value_usdc",  "sum")
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Unique markets: 1,563


,condition_id,question,end_date,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x092471f61558dabaf1ccac4444921db0c974a5734a0dcb55896be0eeff2fd775,"Will Trump say ""Jesus"" this week? (March 8)",2026-03-08 00:00:00+00:00,1300,9966,9.622013e+06,9.619181e+06
1,0x03e38f4c5abeb1560d3af133186dd3130f80183364964abf4724efbc7debeefe,Clippers vs. Warriors,2026-03-03 00:00:00+00:00,5275,44465,7.109552e+06,7.869653e+06
2,0x07ac20a236c76c5d849be442162c57fc5201df6dbda8046d5d917d28c622888a,Pistons vs. Heat,2026-03-08 00:00:00+00:00,3487,24188,3.475741e+06,3.221671e+06
3,0x04cac7412c51f810a0c82d8333d32e6c7107ced282136d8b1751555a99522da8,LoL: Gen.G vs BNK FEARX (BO5) - LCK Cup Playoffs,2026-03-01 00:00:00+00:00,1651,7433,2.479837e+06,2.453234e+06
4,0x0501bc5098a06e1a7e82f9361b3dd9eb3015ff97af7782b1fab5b467e11766d3,LoL: Weibo Gaming vs Anyone's Legend (BO5) - LPL Playoffs,2026-03-05 00:00:00+00:00,2252,17002,2.378354e+06,2.341355e+06
5,0x0a7fa359c73c8ff35786e02f49abdc37b2aa6779a672b8a9b4485f9f26f5ebe3,Pelicans vs. Suns,2026-03-07 00:00:00+00:00,2852,19353,2.309818e+06,2.409969e+06
6,0x0c3061693f0e5dbffabf10ce95632cbd9b2d26c9acd87356b2837fc006560530,Trail Blazers vs. Rockets,2026-03-07 00:00:00+00:00,2786,20363,2.022639e+06,1.983547e+06
7,0x0c4bfb3928b61127182a477ffe9be1a8ceba2ca6a3c2a24e97cc4eda14153189,Spread: Hawks (-6.5),2026-03-07 00:00:00+00:00,564,3786,1.279901e+06,1.305990e+06
8,0x07feac3eb0e4984529e7c7d75043bd35d604c7c1d8a57deab4edbd0f22b96b49,Counter-Strike: MOUZ vs Heroic (BO3) - ESL Pro League Stage 2,2026-03-07 00:00:00+00:00,945,8181,1.219219e+06,1.181134e+06
9,0x01bded8d4eb4719ffcf3b3fd491f30f9726d643cce51f60e9a8b4423c8558da6,Counter-Strike: paiN vs Passion UA (BO3) - ESL Pro League Stage 1,2026-03-03 00:00:00+00:00,1145,11043,1.104741e+06,1.064091e+06


## 7 · Wallet statistics (quantiles)

Distribution of activity and P&L across wallets (training period).

In [35]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "pnl_usdc"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

# number of wallets at or below each quantile threshold
wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])

# append count, mean and std for reference
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")

wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]   = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"]  = wallet_stats["num_markets"].round(1)
wallet_stats["pnl_usdc"]     = wallet_stats["pnl_usdc"].round(2)

wallet_stats

,wallet_count,num_trades,num_markets,pnl_usdc,wallet_count_complement
quantile,,,,,
0.001,29,1.0,1.0,-28747.09,29234
0.01,293,1.0,1.0,-3026.41,28970
0.05,1463,1.0,1.0,-486.22,27800
0.1,2926,1.0,1.0,-132.78,26337
0.25,7316,1.0,1.0,-11.01,21947
0.5,14632,3.0,1.0,0.08,14631
0.75,21947,7.0,3.0,10.64,7316
0.9,26337,23.0,6.0,113.75,2926
0.95,27800,48.0,9.0,472.58,1463


## 8 · Full enriched trade table

One row per wallet × market × timestamp, with all enrichment columns.

In [36]:
DISPLAY_COLS = [
    "wallet", "condition_id", "dt",
    "side", "outcome", "position", "total_quantity", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "num_fills",
]

grouped[DISPLAY_COLS]

,wallet,condition_id,dt,side,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,num_fills
0,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x03e38f4c5abeb1560d3af133186dd3130f80183364964abf4724efbc7debeefe,2026-03-03 01:25:43+00:00,BUY,Clippers,10.000000,10.000000,0.510000,True,1.0,5.100000,10.000000,4.900000,1
1,0x00000db3f2fb2086866753b6e41e426a5312e225,0x06dba2eea29f5a283939f87dfb04d50eaeb35f8bad761ad16037ded5a8a2cfc5,2026-03-01 22:57:31+00:00,BUY,Michigan State Spartans,50.000000,50.000000,0.999000,True,1.0,49.950000,50.000000,0.050000,1
2,0x00000db3f2fb2086866753b6e41e426a5312e225,0x0b8c63df9d7e33b5dd98116f4c04e2eaf7e83df5d1e2e3368f5f5723f5b2e7ac,2026-03-01 23:14:27+00:00,BUY,Nets,50.000000,50.000000,0.983333,True,1.0,49.400000,50.000000,0.600000,3
3,0x0000769690190af200538d3b39f3af22ad2d296f,0x0f02a01099649c28962633ea6d0ad1db994be598089be058ee3366c3df3ed18c,2026-03-03 12:49:17+00:00,BUY,FORZE Reload,132.045625,132.045625,0.152000,False,0.0,20.000000,0.000000,-20.000000,5
4,0x0001c7d116fd33b0edc523b56d36d9a2f47b93e0,0x07ac20a236c76c5d849be442162c57fc5201df6dbda8046d5d917d28c622888a,2026-03-08 00:04:53+00:00,BUY,Pistons,18.300000,18.300000,0.590000,False,0.0,10.797000,0.000000,-10.797000,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
455432,0xffff6537da5efd428cc09187f90cbd2e726d4413,0x0c3061693f0e5dbffabf10ce95632cbd9b2d26c9acd87356b2837fc006560530,2026-03-07 03:16:37+00:00,SELL,Rockets,17.021119,20.000000,0.990000,True,1.0,19.800000,20.000000,-0.200000,1
455433,0xffff6537da5efd428cc09187f90cbd2e726d4413,0x0c3061693f0e5dbffabf10ce95632cbd9b2d26c9acd87356b2837fc006560530,2026-03-07 03:17:03+00:00,SELL,Rockets,2.978881,20.000000,0.500000,True,1.0,16.880697,17.021119,-0.140422,2
455434,0xffff6537da5efd428cc09187f90cbd2e726d4413,0x0c3061693f0e5dbffabf10ce95632cbd9b2d26c9acd87356b2837fc006560530,2026-03-07 03:22:41+00:00,BUY,Trail Blazers,22.978881,20.000000,0.010000,False,0.0,0.200000,0.000000,-0.200000,1
455435,0xffff6537da5efd428cc09187f90cbd2e726d4413,0x0c3061693f0e5dbffabf10ce95632cbd9b2d26c9acd87356b2837fc006560530,2026-03-07 03:23:33+00:00,BUY,Trail Blazers,32.948481,9.969600,0.010000,False,0.0,0.099696,0.000000,-0.099696,1


## 9 · Export top-5% wallets to parquet shards

Identifies top-5% wallets by P&L on the **training period**, then performs a
second partition-by-partition pass to export only those wallets' trades from the
**full dataset** (training + test) to `data/polygon_trades_processed/*.parquet`.

Each row is a single per-side fill, enriched with `final_value_usdc` and a boolean
`is_train` flag. Output remains partitioned by input shard.


In [37]:
# ── identify top-5% wallets by training-period P&L ───────────────────────────
pnl_threshold = wallet_summary["pnl_usdc"].quantile(0.95)
top_wallets   = set(wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "wallet"])
print(f"P&L threshold (95th pct, training): {pnl_threshold:,.2f} USDC")
print(f"Top-5% wallet count:                {len(top_wallets):,}")

# print deciles of training P&L for top wallets
top_pnl = wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "pnl_usdc"]
for i in range(0, 11):
    q = i / 10
    print(f"  P&L at {q:.0%} pct of top wallets: {top_pnl.quantile(q):,.2f} USDC")

P&L threshold (95th pct, training): 472.58 USDC
Top-5% wallet count:                1,464
  P&L at 0% pct of top wallets: 472.65 USDC
  P&L at 10% pct of top wallets: 546.55 USDC
  P&L at 20% pct of top wallets: 657.24 USDC
  P&L at 30% pct of top wallets: 768.17 USDC
  P&L at 40% pct of top wallets: 942.02 USDC
  P&L at 50% pct of top wallets: 1,112.03 USDC
  P&L at 60% pct of top wallets: 1,263.70 USDC
  P&L at 70% pct of top wallets: 1,584.67 USDC
  P&L at 80% pct of top wallets: 2,497.01 USDC
  P&L at 90% pct of top wallets: 5,439.47 USDC
  P&L at 100% pct of top wallets: 357,999.42 USDC


In [38]:
import shutil

EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "token_id", "outcome",
    "quantity", "price", "usdc_amount", "position",
    "final_value_usdc", "trade_pnl",
    "token_winner", "final_price",
    "tx_hash",
    "is_train",
]

EXPORT_READ_COLS = [
    "tx_hash", "log_index", "block_timestamp", "condition_id", "token_id",
    "outcome", "price", "quantity", "usdc_amount", "position", "wallet", "side",
]

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

total_top_rows = 0
total_top_train_rows = 0
top_trades_preview = None
output_files = []

for i, f in enumerate(trade_files, start=1):
    raw_chunk = pd.read_parquet(f, columns=EXPORT_READ_COLS)
    enriched_chunk = enrich_trade_chunk(raw_chunk)
    if len(enriched_chunk) == 0:
        continue

    top_chunk = enriched_chunk[enriched_chunk["wallet"].isin(top_wallets)].copy()
    if len(top_chunk) == 0:
        continue

    is_buy_fill = top_chunk["side"] == "BUY"
    top_chunk["trade_pnl"] = (
        (top_chunk["final_value_usdc"] - top_chunk["usdc_amount"]).where(
            is_buy_fill,
            top_chunk["usdc_amount"] - top_chunk["final_value_usdc"],
        )
    )
    top_chunk["is_train"] = top_chunk["dt"].dt.date <= END_DATE_TRAIN

    export_chunk = top_chunk[EXPORT_COLS]

    out_file = OUT_DIR / f.name
    export_chunk.to_parquet(out_file, index=False)
    output_files.append(out_file)

    if top_trades_preview is None:
        top_trades_preview = export_chunk.head(5).copy()

    total_top_rows += len(export_chunk)
    total_top_train_rows += int(export_chunk["is_train"].sum())

    if i % 25 == 0 or i == len(trade_files):
        print(
            f"Processed export shard {i}/{len(trade_files)} | "
            f"accumulated rows: {total_top_rows:,} | output shards: {len(output_files):,}"
        )

if not output_files:
    empty_file = OUT_DIR / "part-00000.parquet"
    pd.DataFrame(columns=EXPORT_COLS).to_parquet(empty_file, index=False)
    output_files = [empty_file]

top_trades_count = total_top_rows

print(f"Total expanded rows for top wallets: {top_trades_count:,}")
print(f"  training rows: {total_top_train_rows:,}")
print(f"  test rows:     {top_trades_count - total_top_train_rows:,}")
print(f"Output shards:  {len(output_files):,}")
print(f"\nSaved partitioned output → {OUT_DIR}")


Processed export shard 1/1 | accumulated rows: 141,129 | output shards: 1
Total expanded rows for top wallets: 141,129
  training rows: 122,239
  test rows:     18,890
Output shards:  1

Saved partitioned output → ../../data/polygon_trades_processed


In [39]:
pd.set_option("display.max_colwidth", None)
if top_trades_count == 0:
    print("No top-wallet trades found for current date/train split filters.")
    pd.read_parquet(output_files[0]).head(0)
else:
    top_trades_preview


### Unit test: CTF Exchange contract is not treated as a trader

`0x85d355ef32d62b09d2362184f299a7fc662ee1a4` is the Polymarket CTF Exchange
(on-chain matching contract). It can appear in per-side trade rows as a `wallet`
counterparty, but it is **not** a real trader.

This test always enforces that the exchange address is **not** in `top_wallets`.
It also reports whether the exchange appears in the currently filtered dataset
(appearance depends on the chosen date range).


In [40]:
# ── Unit test: CTF Exchange contract is not selected as a top wallet ──────────
CTF_EXCHANGE = "0x85d355ef32d62b09d2362184f299a7fc662ee1a4"

# Presence in grouped depends on date filtering; this is informational.
exchange_rows = grouped[grouped["wallet"] == CTF_EXCHANGE]
if len(exchange_rows) > 0:
    print(f"INFO  CTF Exchange appears in grouped:  {len(exchange_rows):,} rows")
else:
    print("INFO  CTF Exchange rows not present in current filtered dataset")

# Must always hold regardless of date range.
assert CTF_EXCHANGE not in top_wallets, (
    f"CTF Exchange contract ({CTF_EXCHANGE}) was incorrectly selected as a top wallet. "
    "It is the on-chain matching engine, not a real trader."
)

print(f"PASS  CTF Exchange not in top_wallets: top_wallets has {len(top_wallets):,} entries")


INFO  CTF Exchange rows not present in current filtered dataset
PASS  CTF Exchange not in top_wallets: top_wallets has 1,464 entries
